In [1]:
import random
import json

#### JSON from the editor
```json
{
  "nodes": [
    {
      "width": 240,
      "height": 276,
      "id": "0.05272377425322505-1756500835216",
      "type": "eventType",
      "position": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "data": {
        "box": {
          "newEventVars": {
            "0": [
              "payment reminder"
            ],
            "1": [
              "pay order"
            ]
          },
          "newObjectVars": {
            "0": [
              "orders"
            ]
          },
          "filters": [
            {
              "type": "O2E",
              "object": 0,
              "event": 0,
              "qualifier": null
            },
            {
              "type": "O2E",
              "object": 0,
              "event": 1,
              "qualifier": null
            }
          ],
          "sizeFilters": [],
          "constraints": [
            {
              "type": "Filter",
              "filter": {
                "type": "TimeBetweenEvents",
                "from_event": 0,
                "to_event": 1,
                "min_seconds": 0,
                "max_seconds": null
              }
            }
          ],
          "evVarLabels": {},
          "obVarLabels": {}
        }
      },
      "positionAbsolute": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "selected": true,
      "dragging": false
    }
  ],
  "edges": []
}
```

#### JSON given to the DB
```json
{
  "tree": {
    "nodes": [
      {
         "Box": [
           {
             "newEventVars": { "0": ["payment reminder"], "1": ["pay order"] },
             "newObjectVars": { "0": ["orders"] },
             "filters": [
               { "type": "O2E", "object": 0, "event": 0, "qualifier": null },
               { "type": "O2E", "object": 0, "event": 1, "qualifier": null }
             ],
             "sizeFilters": [],
             "constraints": [
               {
                 "type": "Filter",
                 "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": null }
               }
             ],
             "evVarLabels": {},
             "obVarLabels": {}
           },
           [] // IDs of any connected child nodes in the tree
         ]
      }
    ],
    "edgeNames": []
  },
  "database": "DuckDB", // Target Database Engine (if doing DB translation)
  "table_mappings": { ... }
}
```

In [2]:
patterns = [
    "EVENT_A happens before EVENT_B",
    "Whenever EVENT_A occurs, EVENT_B eventually occurs",
    "After EVENT_A, EVENT_B must not occur",
    "EVENT_A and EVENT_B refer to the same OBJECT",
    "EVENT_B happens within TIME hours after EVENT_A"
]

events = [
    "create order",
    "add item",
    "ship package",
    "pay order",
    "cancel order",
    "approve application",
    "submit application",
    "review application",
    "merge pull request",
    "open issue"
]

objects = [
    "orders",
    "items",
    "applications",
    "pull requests",
    "issues"
]

In [6]:
def generate_example():
    pattern = random.choice(patterns)

    A = random.choice(events)
    # Ensure event A and B are different for better logical examples
    B = random.choice([e for e in events if e != A])
    obj = random.choice(objects)
    t = random.choice([1, 6, 24, 72])

    text = pattern.replace("EVENT_A", A)\
                  .replace("EVENT_B", B)\
                  .replace("OBJECT", obj)\
                  .replace("TIME", str(t))

    # Dynamically inject constraints based on the pattern
    constraints = []
    if "before" in pattern or "eventually" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": None }
        }]
    elif "within TIME hours" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": t * 3600 }
        }]

    ocpq_tree = {
        "tree": {
            "nodes": [
                {
                    "Box": [
                        {
                            "newEventVars": { "0": [A], "1": [B] },
                            "newObjectVars": { "0": [obj] },
                            "filters": [
                                { "type": "O2E", "object": 0, "event": 0, "qualifier": None },
                                { "type": "O2E", "object": 0, "event": 1, "qualifier": None }
                            ],
                            "sizeFilters": [],
                            "constraints": constraints,
                            "evVarLabels": {},
                            "obVarLabels": {}
                        },
                        [] # IDs of any connected child nodes in the tree
                    ]
                }
            ],
            "edgeNames": []
        }
    }

    return {"nl": text, "query": ocpq_tree}

dataset = [generate_example() for _ in range(200)]

# with open("ocpq_dataset.json","w") as f:
#     json.dump(dataset,f,indent=2)

In [7]:
dataset[:5]

[{'nl': 'open issue happens before cancel order',
  'query': {'tree': {'nodes': [{'Box': [{'newEventVars': {'0': ['open issue'],
         '1': ['cancel order']},
        'newObjectVars': {'0': ['issues']},
        'filters': [{'type': 'O2E',
          'object': 0,
          'event': 0,
          'qualifier': None},
         {'type': 'O2E', 'object': 0, 'event': 1, 'qualifier': None}],
        'sizeFilters': [],
        'constraints': [{'type': 'Filter',
          'filter': {'type': 'TimeBetweenEvents',
           'from_event': 0,
           'to_event': 1,
           'min_seconds': 0,
           'max_seconds': None}}],
        'evVarLabels': {},
        'obVarLabels': {}},
       []]}],
    'edgeNames': []}}},
 {'nl': 'submit application happens within 1 hours after ship package',
  'query': {'tree': {'nodes': [{'Box': [{'newEventVars': {'0': ['ship package'],
         '1': ['submit application']},
        'newObjectVars': {'0': ['pull requests']},
        'filters': [{'type': 'O2E',
  